# 04B_MODIS_Extraction
Extract MODIS LST from image_metadata.csv.

In [ ]:

!pip -q install earthengine-api pandas

import ee
import pandas as pd
from google.colab import files

ee.Authenticate()
ee.Initialize(project="heatwave-flood-prediction")

print("Upload image_metadata.csv")
uploaded=files.upload()
fname=list(uploaded.keys())[0]
df=pd.read_csv(fname)

rows=[]

for _,r in df.iterrows():
    pt=ee.Geometry.Point([r["longitude"],r["latitude"]])
    d=ee.Date(r["image_date"])

    img=(ee.ImageCollection("MODIS/061/MOD11A1")
          .filterDate(d,d.advance(1,"day"))
          .first())

    vals=img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pt,
        scale=1000,
        maxPixels=1e9
    ).getInfo()

    day=vals.get("LST_Day_1km",None)
    night=vals.get("LST_Night_1km",None)

    rows.append({
        "event_id":r["event_id"],
        "image_date":r["image_date"],
        "LST_Day_C": None if day is None else day*0.02-273.15,
        "LST_Night_C": None if night is None else night*0.02-273.15
    })

out=pd.DataFrame(rows)
out.to_csv("modis_features.csv",index=False)
files.download("modis_features.csv")
print(out.head())
